# 🔥 Forest Fire Area Prediction
## An End-to-End Machine Learning Pipeline

**Dataset**: Montesinho Natural Park, Portugal — 517 fire records collected between 2000 and 2003  
**Goal**: Given weather and environmental conditions at the time of a fire, predict **how many hectares will burn**  
**Audience**: This notebook assumes Python familiarity but **no prior machine learning experience**. Every concept is explained in plain language before any code appears.

---

| Section | Topic |
|---------|-------|
| **0** | Setup & Data Loading |
| **1** | Injecting Noise (simulating real-world dirty data) |
| **2** | Data Cleaning — duplicates, typos, outliers, imputation |
| **3** | Exploratory Data Analysis (EDA) — understanding the data visually |
| **4** | Feature Engineering & Preprocessing |
| **5** | Train / Dev / Test Split & Cross-Validation |
| **6** | Model Training & Comparison (7 models, from simple to advanced) |
| **7** | Explainability with SHAP — understanding *why* the model predicts what it predicts |
| **8** | Bonus: The Two-Stage Model |

> **How to use this notebook**: Run all cells from top to bottom. Each section begins with a plain-language explanation. Key findings are printed after every plot.

---
## Section 0 — Setup & Data Loading

We start by importing every library we will need. Think of this as gathering all your tools before starting a job.  
We also set a **random seed** (`42`) so that every random operation (shuffling, splitting, etc.) produces the same result every time — this makes the notebook fully **reproducible**.

In [ ]:
# ── If any library is missing, uncomment and run the line below first ──────────
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost shap

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold,
    RandomizedSearchCV, GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import SelectKBest, f_regression

import xgboost as xgb
import shap

# ── Global visual style ────────────────────────────────────────────────────────
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')
sns.set_palette('husl')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

print('All libraries loaded successfully!')
print(f'  pandas {pd.__version__} | numpy {np.__version__} | xgboost {xgb.__version__}')

In [ ]:
# Load the original, clean dataset
df_original = pd.read_csv('forestfires.csv')

print(f'Dataset shape: {df_original.shape[0]} rows x {df_original.shape[1]} columns')
print(f'Columns: {list(df_original.columns)}')
df_original.head(10)

### Understanding the Columns

The dataset comes from the **Canadian Fire Weather Index (FWI) system** — a set of standardised indices computed from daily weather readings to estimate fire danger.

| Column | Type | Meaning |
|--------|------|---------|
| X, Y | integer | Grid coordinates within Montesinho Park (1–9) |
| month | text | Month of the fire (jan – dec) |
| day | text | Day of the week (mon – sun) |
| **FFMC** | float | *Fine Fuel Moisture Code* — how dry twigs and leaf litter are. Higher = more flammable |
| **DMC** | float | *Duff Moisture Code* — moisture in the soil's organic layer. Higher = drier |
| **DC** | float | *Drought Code* — long-term dryness and deep fuel dryness |
| **ISI** | float | *Initial Spread Index* — estimated speed of fire spread (combines wind + FFMC) |
| temp | float | Air temperature (°C) |
| RH | float | Relative humidity (%) |
| wind | float | Wind speed (km/h) |
| rain | float | Rainfall (mm) — mostly zero |
| **area** | float | **TARGET**: burned area in hectares (0 – 1090) |

> **Challenge ahead**: `area` is extremely skewed — most fires burn 0 hectares, and a handful burn hundreds. This makes prediction genuinely difficult.

In [ ]:
# Summary statistics for all numeric columns
# min, 25%, 50%, 75%, max — these five numbers tell the 'story' of each column
print('=== Descriptive Statistics ===')
df_original.describe()

In [ ]:
zero_pct = (df_original['area'] == 0).mean() * 100
print(f'Zero-area fires (no spread): {zero_pct:.1f}% of all records')
print(f'Largest fire: {df_original["area"].max():.2f} ha')
print(f'Mean burned area: {df_original["area"].mean():.2f} ha')
print(f'Median burned area: {df_original["area"].median():.2f} ha')
print()
print('→ The mean is much higher than the median — a sign of a heavily skewed distribution.')
print('  A few huge fires pull the average up. We will handle this with a log transform later.')

---
## Section 1 — Injecting Noise (Simulating Dirty Real-World Data)

The Montesinho dataset is **already clean**. In real life, data collected by field teams rarely arrives in perfect condition. To practise the full data-cleaning workflow, we will deliberately introduce five types of problems:

| Problem | Where | What it simulates |
|---------|--------|------------------|
| **Missing values** | FFMC, temp, RH, DC, area | Sensor failures, unentered forms |
| **Impossible outliers** | temp, wind | Data entry errors (temp=200°C) |
| **Duplicate rows** | 5 random rows | Double-submission of records |
| **Category typos** | month | `'aug '`, `'AUG'`, `'agust'` |
| **Wrong data type** | rain | Values stored as text instead of numbers |

All random choices use `random_state=42`, so you will get the exact same dirty dataset every time.

In [ ]:
# Keep a clean backup — we will use it later for comparison in the imputation step
df_clean_backup = df_original.copy()

df_dirty = df_original.copy()
rng = np.random.default_rng(RANDOM_STATE)
n = len(df_dirty)

# ── 1. Missing values (~10% of selected columns) ───────────────────────────────
for col in ['FFMC', 'temp', 'RH', 'DC', 'area']:
    idx = rng.choice(n, size=int(n * 0.10), replace=False)
    df_dirty.loc[idx, col] = np.nan

# ── 2. Impossible outliers ─────────────────────────────────────────────────────
df_dirty.loc[rng.choice(n, size=5, replace=False), 'temp'] = 200.0  # impossible temperature
df_dirty.loc[rng.choice(n, size=5, replace=False), 'wind'] = -5.0   # negative wind speed

# ── 3. Duplicate rows ──────────────────────────────────────────────────────────
dup_rows = df_dirty.iloc[rng.choice(n, size=5, replace=False)]
df_dirty = pd.concat([df_dirty, dup_rows], ignore_index=True)

# ── 4. Category typos in 'month' ───────────────────────────────────────────────
typos = ['aug ', 'AUG', 'agust', 'SEPT', 'sep ']
typo_idx = rng.choice(len(df_dirty), size=5, replace=False)
for i, idx in enumerate(typo_idx):
    df_dirty.loc[idx, 'month'] = typos[i]

# ── 5. Wrong dtype: some 'rain' values stored as strings ──────────────────────
str_idx = rng.choice(len(df_dirty), size=8, replace=False)
df_dirty.loc[str_idx, 'rain'] = df_dirty.loc[str_idx, 'rain'].astype(str)

# Save the dirty version
df_dirty.to_csv('forestfires_dirty.csv', index=False)

print(f'Original dataset: {df_original.shape}')
print(f'Dirty dataset:    {df_dirty.shape}  (added 5 duplicate rows)')
print(f'\nMissing values injected:')
print(df_dirty.isnull().sum()[df_dirty.isnull().sum() > 0])

In [ ]:
# A quick look at the dirty data — notice the NaN values and extreme temp
print('Sample of dirty data (first 15 rows):')
df_dirty[['month', 'temp', 'wind', 'RH', 'rain', 'area']].head(15)

---
## Section 2 — Data Cleaning

We now load the dirty file and fix every problem systematically. Think of this as a medical check-up for your data — we examine, diagnose, and treat each issue in turn.

### 2.1 — First Look at the Dirty Data

Before fixing anything, get the full picture: column types, counts, and missing values.

In [ ]:
# Load the dirty dataset as if we received it fresh from the field
df = pd.read_csv('forestfires_dirty.csv')

print('=== df.info() ===')
df.info()
print()
print('=== Missing value counts ===')
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: heatmap of missing positions — each row is a sample, each column is a feature
# Yellow = missing, Dark = present
sns.heatmap(df.isnull(), cbar=False, yticklabels=False,
            ax=axes[0], cmap='viridis')
axes[0].set_title('Missing Values Map\n(Yellow = Missing, Dark = Present)', fontsize=12)

# Right: percentage missing per column
pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
pct = pct[pct > 0]
axes[1].bar(pct.index, pct.values, color='coral', edgecolor='white')
axes[1].set_title('% Missing per Column', fontsize=12)
axes[1].set_ylabel('% Missing')
axes[1].set_xticklabels(pct.index, rotation=30, ha='right')
for bar, val in zip(axes[1].patches, pct.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontsize=9)

plt.suptitle('Step 1: Diagnosing Missing Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('→ The left map lets us see whether gaps cluster in specific rows (systematic problem)')
print('  or are scattered randomly. Here they appear random — safe to impute.')
print('→ About 10% missing in FFMC, temp, RH, DC, and area — all imputable.')

### 2.2 — Removing Duplicate Rows

A duplicate row is an exact copy of another row. During model training, duplicates give the model a **false sense of confidence** about those specific data points — it sees them multiple times and treats them as stronger evidence than they deserve.

In [ ]:
n_before = len(df)
n_dups   = df.duplicated().sum()

df = df.drop_duplicates().reset_index(drop=True)

print(f'Duplicate rows found and removed: {n_dups}')
print(f'Rows before: {n_before}  →  Rows after: {len(df)}')

### 2.3 — Fixing Categorical Typos

Machine learning treats `'aug'`, `'AUG'`, and `'aug '` as **three completely different categories**. If August has 5 different spellings, the model learns 5 separate (weak) patterns instead of one strong one. We standardise everything to lowercase with no extra spaces.

In [ ]:
print('Month value counts BEFORE cleaning:')
print(df['month'].value_counts())
print('\n→ Notice: "aug ", "AUG", "agust", "SEPT" are all corrupted versions of real months.')

In [ ]:
# Step 1: strip whitespace and lowercase everything
df['month'] = df['month'].str.strip().str.lower()
df['day']   = df['day'].str.strip().str.lower()

# Step 2: correct known typos using a lookup dictionary
month_corrections = {'agust': 'aug', 'sept': 'sep'}
df['month'] = df['month'].replace(month_corrections)

# Step 3: anything still invalid → mark as NaN (will be handled in imputation)
valid_months = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
valid_days   = ['mon','tue','wed','thu','fri','sat','sun']
df.loc[~df['month'].isin(valid_months), 'month'] = np.nan
df.loc[~df['day'].isin(valid_days),     'day']   = np.nan

print('Month value counts AFTER cleaning:')
print(df['month'].value_counts())
print('\n✓ All months are now clean lowercase strings.')

### 2.4 — Fixing Data Types

Some `rain` values were accidentally stored as **text strings** instead of numbers. When pandas reads a column that mixes numbers and text, it silently converts the whole column to `object` type — and you cannot do math on text. `pd.to_numeric(..., errors='coerce')` converts each value to a number, and anything it cannot convert becomes `NaN` (which we will impute later).

In [ ]:
print(f'rain dtype BEFORE fix: {df["rain"].dtype}')

df['rain'] = pd.to_numeric(df['rain'], errors='coerce')

print(f'rain dtype AFTER fix:  {df["rain"].dtype}')
print(f'NaN values introduced (were strings): {df["rain"].isnull().sum()}')

### 2.5 — Detecting and Handling Outliers

An outlier is a value that is unusually far from the rest. Not all outliers are errors — a 200 ha fire is real and important — but physically **impossible** values (temp = 200°C, wind = −5 km/h) must be corrected.

We use **two detection methods** side by side:
- **IQR method**: flags values more than 1.5× the interquartile range beyond Q1/Q3 — robust to extreme outliers
- **Z-score method**: flags values more than 3 standard deviations from the mean — assumes a roughly normal distribution

**Strategy: cap (clip) rather than delete.** Deleting a row throws away all the other valid information in that row. Capping to a physically plausible limit (e.g., max temp 50°C) keeps the row while correcting the single bad value.

In [ ]:
numeric_cols = ['FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain']
report = []

for col in numeric_cols:
    valid = df[col].dropna()
    Q1, Q3 = valid.quantile(0.25), valid.quantile(0.75)
    IQR = Q3 - Q1
    iqr_flag  = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    z_scores  = np.abs((df[col] - valid.mean()) / valid.std())
    z_flag    = (z_scores > 3).sum()
    report.append({'Column': col, 'IQR outliers': iqr_flag,
                   'Z-score outliers': z_flag,
                   'Min': round(valid.min(),2), 'Max': round(valid.max(),2)})

pd.DataFrame(report).set_index('Column')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    bp = axes[i].boxplot(df[col].dropna(), patch_artist=True, widths=0.5,
                         boxprops=dict(facecolor='lightsteelblue'),
                         medianprops=dict(color='navy', linewidth=2),
                         flierprops=dict(marker='o', color='coral', alpha=0.6))
    axes[i].set_title(col, fontweight='bold', fontsize=11)
    axes[i].set_xticks([])

plt.suptitle('Box Plots — Identifying Outliers\n'
             '(Coral dots beyond the whiskers are potential outliers)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('→ temp: the injected value of 200°C is a clear outlier (well beyond the box).')
print('→ wind: a value of -5 is physically impossible (wind speed cannot be negative).')
print('→ rain: many zeros with a few extreme values — typical for rainfall data.')

In [ ]:
# Clip values to physically plausible ranges
# These bounds come from domain knowledge (what is physically possible)
bounds = {
    'temp': (  2.0,  50.0),   # temperature range for Portugal (°C)
    'wind': (  0.0,  15.0),   # wind speed cannot be negative
    'RH':   (  1.0, 100.0),   # relative humidity is a percentage
    'FFMC': (  0.0, 101.0),   # theoretical FFMC scale
    'ISI':  (  0.0,  60.0),   # ISI rarely exceeds 60 even in extreme conditions
}

for col, (lo, hi) in bounds.items():
    n_capped = ((df[col] < lo) | (df[col] > hi)).sum()
    df[col]  = df[col].clip(lo, hi)
    if n_capped > 0:
        print(f'{col:6s}: {n_capped} values capped to [{lo}, {hi}]')

print('\n✓ Outlier capping complete.')
print('  Note: we cap rather than delete — removing a row throws away all other valid data in it.')

### 2.6 — Imputing Missing Values

**Imputation** means filling in the gaps with plausible estimates. We will demonstrate three strategies and compare them visually, then choose the best one for the rest of the notebook.

| Strategy | How it works | Strength | Weakness |
|----------|-------------|----------|----------|
| **Mean / Median** | Fill with the column average | Instant, simple | Ignores all relationships between columns |
| **KNN Imputer** | Find the 5 most similar rows and average their values | Uses local structure | Slower on large datasets |
| **MICE** *(Iterative Imputer)* | Build a model predicting each missing column from all others; iterate until stable | Most accurate; preserves inter-feature relationships | Slightly more complex internally |

**MICE** stands for *Multiple Imputation by Chained Equations*. It is considered the gold standard for missing data imputation in statistics.

In [ ]:
print('Missing values remaining before imputation:')
missing_now = df.isnull().sum()
print(missing_now[missing_now > 0])

In [ ]:
# To impute with KNN and MICE we need a fully numeric dataframe.
# We encode month and day as integers, impute, then decode them back.

MONTH_ORDER = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
DAY_ORDER   = ['mon','tue','wed','thu','fri','sat','sun']

month_map = {m: i for i, m in enumerate(MONTH_ORDER)}  # jan->0, ..., dec->11
day_map   = {d: i for i, d in enumerate(DAY_ORDER)}    # mon->0, ..., sun->6
month_inv = {v: k for k, v in month_map.items()}
day_inv   = {v: k for k, v in day_map.items()}

# Fill categorical NaNs with mode before numeric encoding
df_for_impute = df.copy()
df_for_impute['month'] = df_for_impute['month'].fillna(df_for_impute['month'].mode()[0])
df_for_impute['day']   = df_for_impute['day'].fillna(df_for_impute['day'].mode()[0])

df_for_impute['month_num'] = df_for_impute['month'].map(month_map)
df_for_impute['day_num']   = df_for_impute['day'].map(day_map)
df_for_impute = df_for_impute.drop(columns=['month', 'day'])

print(f'Numeric dataframe ready for imputation: {df_for_impute.shape}')
print(f'Still missing: {df_for_impute.isnull().sum().sum()} values')

In [ ]:
# ── Strategy 1: Median Fill ────────────────────────────────────────────────────
df_median = df_for_impute.copy()
for col in df_median.columns:
    df_median[col] = df_median[col].fillna(df_median[col].median())
print('Strategy 1 (Median Fill): Done')

# ── Strategy 2: KNN Imputer ────────────────────────────────────────────────────
knn_imputer = KNNImputer(n_neighbors=5)
df_knn = pd.DataFrame(knn_imputer.fit_transform(df_for_impute),
                      columns=df_for_impute.columns)
print('Strategy 2 (KNN, k=5): Done')

# ── Strategy 3: MICE / Iterative Imputer ──────────────────────────────────────
mice = IterativeImputer(max_iter=10, random_state=RANDOM_STATE)
df_mice = pd.DataFrame(mice.fit_transform(df_for_impute),
                       columns=df_for_impute.columns)
print('Strategy 3 (MICE / Iterative Imputer): Done')
print('\n→ All three strategies complete. Now let us compare how well they preserve the data distribution.')

In [ ]:
# Compare distributions of 'temp' before and after each strategy
# A good imputation strategy should barely change the shape of the distribution

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
ref_col = 'temp'
original_vals = df_clean_backup[ref_col]

for ax, (label, imp_df) in zip(axes, [
    ('Strategy 1: Median Fill', df_median),
    ('Strategy 2: KNN Imputer', df_knn),
    ('Strategy 3: MICE', df_mice),
]):
    original_vals.plot.kde(ax=ax, label='Original', color='navy',
                           linewidth=2.5)
    imp_df[ref_col].plot.kde(ax=ax, label=label.split(':')[0],
                              color='coral', linewidth=2, linestyle='--')
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel('Temperature (°C)')
    ax.set_xlim(-5, 45)
    ax.legend(fontsize=8)

plt.suptitle(f'Distribution of `{ref_col}` — Original vs Imputed\n'
             '(Dashed line should closely follow the solid line)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print('→ Median Fill: creates a sharp spike at the median value — distorts the distribution.')
print('→ KNN Imputer: smoother result, closer to the original shape.')
print('→ MICE:        generally the best match — we will use this for the rest of the notebook.')

In [ ]:
# ── Build the final clean dataframe using MICE results ─────────────────────────
df_clean = df_mice.copy()

# Restore month and day string labels from the imputed numeric values
df_clean['month'] = df_clean['month_num'].clip(0, 11).round().astype(int).map(month_inv)
df_clean['day']   = df_clean['day_num'].clip(0, 6).round().astype(int).map(day_inv)
df_clean = df_clean.drop(columns=['month_num', 'day_num'])

# Clip area to 0 — imputation may occasionally produce tiny negative values
df_clean['area'] = df_clean['area'].clip(lower=0)

# Round X and Y back to integers (they are grid coordinates)
df_clean['X'] = df_clean['X'].round().astype(int)
df_clean['Y'] = df_clean['Y'].round().astype(int)

print(f'✅ Clean dataset ready: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns')
print(f'   Missing values remaining: {df_clean.isnull().sum().sum()}')
df_clean.head()

---
## Section 3 — Exploratory Data Analysis (EDA)

EDA is the process of **getting to know your data** before building any model. We use plots and statistics to answer questions like:
- How is each feature distributed?
- Which features correlate with the target?
- Are there any patterns we should be aware of?

Good EDA prevents surprises later and informs better modelling decisions.

### 3.1 — Univariate Analysis
*(Looking at one variable at a time)*

In [ ]:
num_cols = ['FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain', 'area']

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df_clean[col], bins=35, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontweight='bold', fontsize=12)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')
    mean_val = df_clean[col].mean()
    axes[i].axvline(mean_val, color='coral', linewidth=1.5, linestyle='--', label=f'Mean={mean_val:.1f}')
    axes[i].legend(fontsize=8)

plt.suptitle('Distribution of All Numeric Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Key observations:')
print('  FFMC: concentrated at high values (>85) — most fires happen when fuels are very dry')
print('  DMC/DC: wide spread — reflect accumulated seasonal dryness')
print('  rain: almost entirely zero — rain is rare during fire conditions')
print('  area: extreme right skew — the single most important feature to address before modelling')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

month_counts = df_clean['month'].value_counts().reindex(MONTH_ORDER).fillna(0)
axes[0].bar(MONTH_ORDER, month_counts.values, color='coral', edgecolor='white')
axes[0].set_title('Number of Fires by Month', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Count')
axes[0].axvspan(7.5, 8.5, alpha=0.15, color='red', label='Aug/Sep peak')
axes[0].legend()

day_counts = df_clean['day'].value_counts().reindex(DAY_ORDER).fillna(0)
axes[1].bar(DAY_ORDER, day_counts.values, color='teal', edgecolor='white')
axes[1].set_title('Number of Fires by Day of Week', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print('→ August and September account for the majority of fires — Portugal summer is very dry.')
print('→ Fires are slightly more frequent on weekends — possibly more human activity.')

### 3.2 — Deep-Dive into the Target Variable (`area`)

The burned area is our prediction target. Understanding its distribution is critical — it tells us how to prepare it for modelling.

**The Problem**: The raw `area` is *zero-inflated* (many zeros) and *right-skewed* (a few enormous values). Models struggle with this because the loss function gets dominated by a handful of extreme fires.

**The Solution**: Apply a **log(area + 1)** transformation. The `+1` ensures that area=0 maps to log(1)=0 (not to −∞). After the transform, the distribution is much more bell-shaped and the model can learn from all fires, not just the large ones.

In [ ]:
# Add log-transformed target — we will use this throughout modelling
df_clean['log_area'] = np.log1p(df_clean['area'])  # log1p(x) = log(x + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Raw area
axes[0].hist(df_clean['area'], bins=40, color='salmon', edgecolor='white')
axes[0].set_title('Raw area (hectares)', fontweight='bold')
axes[0].set_xlabel('Hectares')
z_pct = (df_clean['area'] == 0).mean() * 100
axes[0].text(0.55, 0.85, f'{z_pct:.0f}% are zero', transform=axes[0].transAxes,
             fontsize=11, color='navy', fontweight='bold')

# Log-transformed — zoomed in
axes[1].hist(df_clean['log_area'], bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('log(area + 1) — Log Transform', fontweight='bold')
axes[1].set_xlabel('log(1 + hectares)')

# Scatter: temp vs log_area
scatter = axes[2].scatter(df_clean['temp'], df_clean['log_area'],
                          c=df_clean['log_area'], cmap='YlOrRd', alpha=0.5, edgecolors='none')
axes[2].set_title('Temperature vs log(area+1)', fontweight='bold')
axes[2].set_xlabel('Temperature (°C)')
axes[2].set_ylabel('log(area+1)')
plt.colorbar(scatter, ax=axes[2], label='log(area+1)')

plt.suptitle('Target Variable Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Raw area  — Skewness: {df_clean["area"].skew():.2f}  (skewness > 1 = strongly right-skewed)')
print(f'Log area  — Skewness: {df_clean["log_area"].skew():.2f}  (much closer to 0 = more symmetric)')

### 3.3 — Correlation Analysis

A **correlation matrix** shows how strongly pairs of variables move together:
- **+1.0** = when one goes up, the other always goes up (perfect positive correlation)
- **−1.0** = when one goes up, the other always goes down (perfect negative correlation)
- **0.0** = no relationship

We use **Pearson** correlation for general relationships, and **Spearman rank** correlation for the skewed log-target (Spearman is more robust to outliers).

In [ ]:
corr_cols = ['FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain', 'area']
corr_mat  = df_clean[corr_cols].corr(method='pearson')
mask      = np.triu(np.ones_like(corr_mat, dtype=bool))  # hide the upper triangle

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax,
            linewidths=0.5, annot_kws={'size': 10},
            vmin=-1, vmax=1)
ax.set_title('Pearson Correlation Matrix\n'
             '(Each cell = correlation between two variables)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key findings:')
print('  DMC & DC:    r = {:.2f} — both measure long-term dryness (expected to be correlated)'.format(
      corr_mat.loc['DMC','DC']))
print('  temp & RH:   r = {:.2f} — hotter days tend to have lower humidity'.format(
      corr_mat.loc['temp','RH']))
print('  ISI & FFMC:  r = {:.2f} — ISI partially depends on FFMC by definition'.format(
      corr_mat.loc['ISI','FFMC']))
print('  area with anything: all correlations are weak — fire spread is genuinely hard to predict!')

In [ ]:
# Spearman correlation of each feature with the log-transformed target
spearman_target = df_clean[corr_cols + ['log_area']].corr(
    method='spearman')['log_area'].drop('log_area')
spearman_sorted = spearman_target.abs().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['salmon' if v < 0 else 'steelblue'
          for v in spearman_target[spearman_sorted.index]]
bars = ax.barh(spearman_sorted.index,
               spearman_target[spearman_sorted.index],
               color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Spearman Correlation with log(area+1)\n'
             '(Blue = higher feature → more fire, Red = higher feature → less fire)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Spearman Correlation Coefficient')
plt.tight_layout()
plt.show()

print('→ Spearman measures rank-order correlation — much more appropriate for skewed data.')
print('→ Even with the log transform, the strongest correlations are only around ±0.2.')
print('  This confirms: predicting fire area from these features alone is a hard problem.')

### 3.4 — Feature–Target Relationships

Box plots grouped by categorical variables help us see whether month or day of week have any effect on fire size. We use log(area+1) on the y-axis so the extreme outliers don't collapse all the boxes to the bottom.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(data=df_clean, x='month', y='log_area',
            order=MONTH_ORDER, ax=axes[0], palette='husl')
axes[0].set_title('log(area+1) by Month', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('log(1 + ha burned)')
axes[0].tick_params(axis='x', rotation=30)

sns.boxplot(data=df_clean, x='day', y='log_area',
            order=DAY_ORDER, ax=axes[1], palette='husl')
axes[1].set_title('log(area+1) by Day of Week', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('log(1 + ha burned)')

plt.suptitle('Do Month and Day Affect Fire Size?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('→ August and September tend to have higher median burned area (taller boxes).')
print('→ Day of week shows very little variation — fires burn equally on any day of the week.')

### 3.5 — Spatial Analysis

The dataset includes X and Y grid coordinates (1–9) within the Montesinho park map. By aggregating fire counts and areas by location, we can identify **hotspot zones**. Note that a zone with many fires is not necessarily the same as a zone with large fires.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Fire count per grid cell
count_pivot = df_clean.pivot_table(
    values='area', index='Y', columns='X', aggfunc='count', fill_value=0)
sns.heatmap(count_pivot, ax=axes[0], cmap='YlOrRd',
            annot=True, fmt='d', linewidths=0.3,
            cbar_kws={'label': 'Number of fires'})
axes[0].set_title('Number of Fires\nby Park Grid Location', fontsize=12, fontweight='bold')
axes[0].invert_yaxis()
axes[0].set_xlabel('X coordinate')
axes[0].set_ylabel('Y coordinate')

# Median log(area) per grid cell
area_pivot = df_clean.pivot_table(
    values='log_area', index='Y', columns='X', aggfunc='median', fill_value=0)
sns.heatmap(area_pivot, ax=axes[1], cmap='YlOrRd',
            annot=True, fmt='.1f', linewidths=0.3,
            cbar_kws={'label': 'Median log(area+1)'})
axes[1].set_title('Median log(area+1)\nby Park Grid Location', fontsize=12, fontweight='bold')
axes[1].invert_yaxis()
axes[1].set_xlabel('X coordinate')
axes[1].set_ylabel('Y coordinate')

plt.suptitle('Spatial Fire Map — Montesinho Park', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('→ Some zones (e.g., X=8, Y=6) have many fires but not necessarily the largest.')
print('→ Zones with few fire records may reflect low access or different vegetation type.')
print('→ The spatial information (X, Y) may help the model but may also overfit on small zones.')

---
## Section 4 — Feature Engineering & Preprocessing

Before feeding data into a machine learning model, we need to transform it into a purely numeric format that the model can process. This section covers:
1. Applying the log transform to the target
2. Encoding categorical columns (month, day) in two ways
3. Scaling numeric features (required for some models)
4. A quick feature importance preview

### 4.1 — Target Transform (Already Done Above)

We added `df_clean['log_area'] = log(area + 1)` in Section 3. This is our **modelling target** throughout Section 6. When we report final results, we will invert this transform (`exp(prediction) − 1`) to get predictions back in hectares.

### 4.2 — Encoding Categorical Variables: One-Hot vs Cyclic

Machine learning models need **numbers**, not text. We have two ways to convert `month` and `day`:

**Option A — One-Hot Encoding** (our primary choice)
Creates a 0/1 column for each category. `month='aug'` becomes `month_aug=1` and all other month columns = 0. Simple and transparent.

**Option B — Cyclic Encoding** (shown as an educational bonus)
Converts months to a circle using sine and cosine, so December (month 12) and January (month 1) end up close to each other — which they are in real life.

In [ ]:
# ── Option A: One-Hot Encoding ─────────────────────────────────────────────────
df_ohe = pd.get_dummies(df_clean, columns=['month', 'day'], drop_first=False)

print(f'Shape BEFORE one-hot encoding: {df_clean.shape}')
print(f'Shape AFTER one-hot encoding:  {df_ohe.shape}')
new_cols = [c for c in df_ohe.columns if c not in df_clean.columns]
print(f'\nNew binary columns created ({len(new_cols)} total):')
print(new_cols)

In [ ]:
# ── Option B: Cyclic Encoding (bonus — educational only) ──────────────────────
# Months 1-12 are mapped onto a circle using sin and cos
# This way the model knows Jan and Dec are adjacent, not 11 months apart

month_num_map = {m: i+1 for i, m in enumerate(MONTH_ORDER)}
day_num_map   = {d: i+1 for i, d in enumerate(DAY_ORDER)}

df_cyclic = df_clean.copy()
df_cyclic['month_sin'] = np.sin(2*np.pi * df_cyclic['month'].map(month_num_map) / 12)
df_cyclic['month_cos'] = np.cos(2*np.pi * df_cyclic['month'].map(month_num_map) / 12)
df_cyclic['day_sin']   = np.sin(2*np.pi * df_cyclic['day'].map(day_num_map) / 7)
df_cyclic['day_cos']   = np.cos(2*np.pi * df_cyclic['day'].map(day_num_map) / 7)
df_cyclic = df_cyclic.drop(columns=['month', 'day'])

# Visualise what cyclic encoding looks like
fig, ax = plt.subplots(figsize=(6, 6))
mn_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
for i, name in enumerate(mn_labels):
    angle = 2 * np.pi * (i+1) / 12
    cx, cy = np.cos(angle), np.sin(angle)
    ax.plot(cx, cy, 'o', markersize=14, color='steelblue', zorder=3)
    ax.annotate(name, (cx*1.15, cy*1.15), ha='center', va='center', fontsize=9)
theta = np.linspace(0, 2*np.pi, 300)
ax.plot(np.cos(theta), np.sin(theta), 'lightgray', zorder=1)
ax.set_xlim(-1.4, 1.4)
ax.set_ylim(-1.4, 1.4)
ax.set_aspect('equal')
ax.set_title('Cyclic Encoding of Month\nDec and Jan are adjacent on the circle', fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()

print('Cyclic encoding uses just 2 columns (sin + cos) instead of 12 one-hot columns.')
print('It is theoretically better for models that care about distance between values')
print('(like SVR or Ridge). For tree-based models, one-hot is usually just as good.')

#### Encoding Strategy Comparison

| | One-Hot Encoding | Cyclic Encoding |
|---|---|---|
| Intuitive? | ✅ Very easy to understand | ❌ Requires explanation |
| Preserves seasonality (Dec ≈ Jan)? | ❌ No | ✅ Yes |
| Columns added | 19 (12 months + 7 days) | 4 (sin/cos for each) |
| Works with tree models? | ✅ | ✅ |
| Works with SVR / Ridge? | ✅ | ✅ (theoretically better) |

**Decision**: We will use **One-Hot Encoding** for all models — it is more transparent for a mixed audience and performs well with tree-based models (the majority of what we build).

In [ ]:
# ── Set up the feature matrix (X) and target (y) ──────────────────────────────
FEATURE_COLS = [c for c in df_ohe.columns if c not in ['area', 'log_area']]

X     = df_ohe[FEATURE_COLS].copy()
y     = df_ohe['log_area'].copy()    # log-transformed target (used for training)
y_raw = df_ohe['area'].copy()         # raw target in hectares (used for final reporting)

# Cast boolean one-hot columns to int (some sklearn estimators prefer this)
bool_cols = X.select_dtypes(include='bool').columns
X[bool_cols] = X[bool_cols].astype(int)

print(f'Feature matrix X: {X.shape[0]} rows x {X.shape[1]} columns')
print(f'Target y (log_area): range [{y.min():.3f}, {y.max():.3f}]')
print(f'Target y_raw (area): range [{y_raw.min():.2f}, {y_raw.max():.2f}] ha')

### 4.3 — Feature Importance Preview (SelectKBest)

Before building complex models, we can get a quick sense of which features have the strongest **linear** relationship with our target using a statistical F-test. This is just a preview — our models will compute more sophisticated importance scores later.

In [ ]:
selector = SelectKBest(score_func=f_regression, k='all')
selector.fit(X, y)

importance_df = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'F-Score': selector.scores_,
}).sort_values('F-Score', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
top_n = 15
top = importance_df.head(top_n)
bars = ax.barh(top['Feature'][::-1], top['F-Score'][::-1],
               color='steelblue', edgecolor='white')
ax.set_title(f'Top {top_n} Features by F-Score (linear relationship with log area)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('F-Score (higher = stronger linear relationship with target)')
plt.tight_layout()
plt.show()

print('Top 5 features by linear F-score:')
for _, row in importance_df.head().iterrows():
    print(f'  {row["Feature"]:20s}  F={row["F-Score"]:.2f}')
print()
print('Note: F-score only measures LINEAR relationships.')
print('Tree models will discover non-linear interactions that this test cannot detect.')

---
## Section 5 — Train / Dev / Test Split & Cross-Validation

### 5.1 — The Three-Way Split

We split the data into three non-overlapping sets:

| Set | Size | Purpose |
|-----|------|---------|
| **Train** | 70% | The model learns from this |
| **Dev** (validation) | 10% | We tune hyperparameters using this — the model never trains on it |
| **Test** | 20% | The final, untouched report card — used **once**, at the very end |

**Why not just train/test?** If you tune your model based on the test set, you are indirectly fitting to it, and your final score will be over-optimistic. The dev set gives you a safe tuning ground.

We **stratify** by fire/no-fire (area > 0) to ensure all three sets have the same proportion of zero-area records.

In [ ]:
# Stratification flag: did the fire spread at all?
fire_flag = (y_raw > 0).astype(int)

# Step 1: carve out the 20% test set
X_traindev, X_test, y_traindev, y_test, yr_traindev, yr_test = train_test_split(
    X, y, y_raw, test_size=0.20, random_state=RANDOM_STATE, stratify=fire_flag
)

# Step 2: from the remaining 80%, split out 10% dev (= 12.5% of the 80%)
ff_traindev = (yr_traindev > 0).astype(int)
X_train, X_dev, y_train, y_dev, yr_train, yr_dev = train_test_split(
    X_traindev, y_traindev, yr_traindev,
    test_size=0.125, random_state=RANDOM_STATE, stratify=ff_traindev
)

print(f'Training set :  {len(X_train):3d} rows  ({len(X_train)/len(X)*100:.1f}%)')
print(f'Dev set      :  {len(X_dev):3d} rows  ({len(X_dev)/len(X)*100:.1f}%)')
print(f'Test set     :  {len(X_test):3d} rows  ({len(X_test)/len(X)*100:.1f}%)')
print()
print('Fire proportions (% of rows with area > 0):')
print(f'  Train: {(yr_train > 0).mean()*100:.1f}%')
print(f'  Dev:   {(yr_dev   > 0).mean()*100:.1f}%')
print(f'  Test:  {(yr_test  > 0).mean()*100:.1f}%')
print('\n→ Stratification worked — all three sets have the same fire/no-fire ratio.')

In [ ]:
# Scale features — REQUIRED for SVR, Ridge, and Lasso (distance/magnitude-sensitive models)
# Tree-based models (Decision Tree, Random Forest, XGBoost) do NOT need scaling
# IMPORTANT: fit the scaler ONLY on training data, then transform dev and test
#            If you fit on all data first, you leak information about dev/test into the scaler

scaler        = StandardScaler()
X_train_sc    = scaler.fit_transform(X_train)   # fit + transform
X_dev_sc      = scaler.transform(X_dev)          # transform only
X_test_sc     = scaler.transform(X_test)         # transform only

print('StandardScaler fitted on training data only.')
print(f'Training feature means (first 5): {scaler.mean_[:5].round(3)}')
print()
print('Why this matters: if we scale the entire dataset first, the scaler has already')
print('"seen" the test data (it used their values to compute the mean and std).')
print('This is called data leakage — it makes test results unrealistically good.')

### 5.2 — 5-Fold Cross-Validation

**K-Fold Cross-Validation** gives a more reliable estimate of model performance than a single train/val split. Here's how it works:

1. Split the training data into 5 equal *folds*
2. Train on folds 1–4, evaluate on fold 5
3. Train on folds 1,2,3,5, evaluate on fold 4
4. ... repeat until every fold has been the validation set once
5. Average the 5 scores — this is a more stable estimate than any single split

We will use this inside `evaluate_model()` in Section 6 for every model.

In [ ]:
# Demonstrate 5-fold CV with a simple decision tree
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
demo_model = DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE)

cv_scores = cross_val_score(
    demo_model, X_train, y_train,
    cv=kf, scoring='neg_root_mean_squared_error'
)
rmse_folds = -cv_scores

print('5-Fold CV Results (Decision Tree, max_depth=4):')
for i, score in enumerate(rmse_folds, 1):
    bar = '=' * int(score * 15)
    print(f'  Fold {i}: RMSE = {score:.4f}  |{bar}')
print(f'\n  Mean RMSE : {rmse_folds.mean():.4f}')
print(f'  Std  RMSE : {rmse_folds.std():.4f}  (low std = consistent performance across folds)')
print()
print('→ We want BOTH a low mean (accurate) and a low std (stable).')
print('  A high std means the model is sensitive to which data it sees — a sign of instability.')

---
## Section 6 — Model Training & Comparison

We will train **7 models**, starting from the simplest and progressing to the most powerful. For each model we report:

| Metric | What it measures |
|--------|------------------|
| **CV RMSE** | Average root-mean-squared error across 5 folds (in log scale) — used for tuning |
| **Test RMSE (ha)** | RMSE on the test set, in original hectares — penalises large errors heavily |
| **Test MAE (ha)** | Mean absolute error on test set — average size of our mistakes in hectares |
| **R²** | Fraction of variance explained. 1.0 = perfect, 0.0 = as good as guessing the mean |

> **Important note on RMSE vs MAE for this dataset**: Because of the single giant fire (1090 ha), RMSE will be high for *all* models — even the naive mean predictor. MAE is the more honest metric here.

In [ ]:
# Storage for all model results
model_results = {}

def evaluate_model(name, model, Xtr, ytr, Xte, yte, yte_raw, n_folds=5):
    """
    Train a model, run 5-fold CV, evaluate on the test set.
    Results are stored in model_results[name] and printed.

    Parameters
    ----------
    name    : label for the model (shown in plots and tables)
    model   : an unfitted sklearn-compatible estimator
    Xtr     : training feature matrix (scaled or unscaled as appropriate)
    ytr     : training target (log-transformed)
    Xte     : test feature matrix
    yte     : test target (log-transformed, for CV scoring)
    yte_raw : test target in original hectares (for reporting)
    """
    # 5-fold CV on the training set
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    cv_rmse = -cross_val_score(model, Xtr, ytr, cv=kf,
                                scoring='neg_root_mean_squared_error')

    # Train on the full training set
    model.fit(Xtr, ytr)

    # Predict on the test set, then invert the log transform
    y_pred_log = model.predict(Xte)
    y_pred_ha  = np.expm1(y_pred_log).clip(min=0)  # clip negatives to 0

    rmse = np.sqrt(mean_squared_error(yte_raw, y_pred_ha))
    mae  = mean_absolute_error(yte_raw, y_pred_ha)
    r2   = r2_score(yte_raw, y_pred_ha)

    model_results[name] = {
        'CV RMSE (log)':  f'{cv_rmse.mean():.3f} ± {cv_rmse.std():.3f}',
        'Test RMSE (ha)': round(rmse, 2),
        'Test MAE (ha)':  round(mae,  2),
        'R2':             round(r2,   3),
        '_model':         model,
        '_preds':         y_pred_ha,
    }

    print(f'{'─'*50}')
    print(f'{name}')
    print(f'  5-Fold CV RMSE (log) : {cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}')
    print(f'  Test RMSE (ha)       : {rmse:.2f}')
    print(f'  Test MAE  (ha)       : {mae:.2f}')
    print(f'  R²                   : {r2:.3f}')
    return model

### 6.1 — Baseline: Naive Mean Predictor

The simplest possible "model" just predicts the same value for every fire: the **mean of the training set**. This is our baseline — any real model that cannot beat it is useless. The R² of the mean predictor is exactly 0 by definition.

In [ ]:
# Naive mean: always predict the training mean, converted back to hectares
mean_log_pred = y_train.mean()
naive_ha_pred = np.full(len(yr_test), np.expm1(mean_log_pred))

naive_rmse = np.sqrt(mean_squared_error(yr_test, naive_ha_pred))
naive_mae  = mean_absolute_error(yr_test, naive_ha_pred)
naive_r2   = r2_score(yr_test, naive_ha_pred)

model_results['Naive Mean'] = {
    'CV RMSE (log)':  'N/A',
    'Test RMSE (ha)': round(naive_rmse, 2),
    'Test MAE (ha)':  round(naive_mae,  2),
    'R2':             round(naive_r2,   3),
    '_preds':         naive_ha_pred,
}

print('Naive Mean Predictor')
print(f'  Constant prediction  : {np.expm1(mean_log_pred):.2f} ha')
print(f'  Test RMSE (ha)       : {naive_rmse:.2f}')
print(f'  Test MAE  (ha)       : {naive_mae:.2f}')
print(f'  R²                   : {naive_r2:.3f}')
print()
print('Note: the naive mean often has a BETTER RMSE than complex models on this dataset.')
print('Why? The single 1090-ha fire drives RMSE extremely high for any model that tries to')
print('predict it. The mean predictor "gives up" on it and wins by default on RMSE.')
print('This is why we also report MAE — it is more robust to that extreme outlier.')

### 6.2 — Decision Tree Regressor

A decision tree asks a series of yes/no questions about the features to reach a prediction — like a flowchart. For example: *"Is temp > 20°C? → Yes → Is FFMC > 90? → Yes → Predict 3.2 ha"*

**Problem**: Without constraints, a tree will memorise the entire training set (every row becomes its own leaf). This is called **overfitting** — the model performs well on training data but poorly on new data. We control this with `max_depth`.

In [ ]:
# Demonstrate overfitting with an unlimited tree
dt_unlimited = DecisionTreeRegressor(random_state=RANDOM_STATE)
dt_unlimited.fit(X_train, y_train)

train_rmse_unl = np.sqrt(mean_squared_error(y_train, dt_unlimited.predict(X_train)))
dev_rmse_unl   = np.sqrt(mean_squared_error(y_dev,   dt_unlimited.predict(X_dev)))

print('Unlimited Decision Tree (no max_depth constraint):')
print(f'  Train RMSE: {train_rmse_unl:.4f}  ← near zero: memorised the training data')
print(f'  Dev   RMSE: {dev_rmse_unl:.4f}  ← much higher: fails on new data')
print()
print('This gap (train ≈ 0, dev >> 0) is the classic signature of overfitting.')

In [ ]:
# Find the best max_depth by comparing train vs dev RMSE
depths       = [2, 3, 4, 5, 6, 7, 8, None]
train_errs   = []
dev_errs     = []

for d in depths:
    dt = DecisionTreeRegressor(max_depth=d, random_state=RANDOM_STATE)
    dt.fit(X_train, y_train)
    train_errs.append(np.sqrt(mean_squared_error(y_train, dt.predict(X_train))))
    dev_errs.append(np.sqrt(mean_squared_error(y_dev, dt.predict(X_dev))))

depth_labels = [str(d) if d else 'None' for d in depths]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(depth_labels, train_errs, 'o-', label='Train RMSE', color='steelblue', linewidth=2)
ax.plot(depth_labels, dev_errs,   's-', label='Dev RMSE',   color='coral',     linewidth=2)
best_idx = int(np.argmin(dev_errs))
ax.axvline(best_idx, color='green', linestyle='--', alpha=0.6,
           label=f'Best depth = {depth_labels[best_idx]}')
ax.set_xlabel('max_depth')
ax.set_ylabel('RMSE (log scale)')
ax.set_title('Decision Tree: Depth vs RMSE\n'
             'Underfitting ← | → Overfitting', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

best_depth = depths[best_idx]
print(f'Best max_depth on dev set: {best_depth}')
print('→ Left of the valley = underfitting (too simple to learn the patterns)')
print('→ Right of the valley = overfitting (too complex, memorises noise)')
print('→ The valley = the sweet spot we want')

In [ ]:
# Train final decision tree and evaluate
dt_final = evaluate_model(
    'Decision Tree',
    DecisionTreeRegressor(max_depth=best_depth, random_state=RANDOM_STATE),
    X_train, y_train, X_test, y_test, yr_test
)

# Visualise the top 3 levels of the tree
fig, ax = plt.subplots(figsize=(20, 6))
plot_tree(dt_final, feature_names=FEATURE_COLS, filled=True,
          rounded=True, max_depth=3, ax=ax, fontsize=8,
          impurity=False, proportion=False)
ax.set_title('Decision Tree — First 3 Levels\n'
             '(Blue = small prediction, Orange = large prediction)', fontsize=12)
plt.tight_layout()
plt.show()

### 6.3 — Ridge and Lasso (Regularised Linear Regression)

Linear regression fits a straight-line equation: `prediction = w1*FFMC + w2*temp + ... + bias`. The model learns the weights `w1, w2, ...`.

**Ridge (L2 regularisation)** adds a penalty for large weights, discouraging the model from relying too heavily on any single feature. Good when all features are somewhat useful.

**Lasso (L1 regularisation)** can shrink some weights all the way to **zero**, effectively removing those features. Good for automatic feature selection.

The `alpha` parameter controls how strong the penalty is. We use `GridSearchCV` to find the best `alpha` on the training set.

In [ ]:
# ── Ridge ──────────────────────────────────────────────────────────────────────
alphas = [0.01, 0.1, 1.0, 5.0, 10.0, 50.0, 100.0]
kf5    = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

ridge_gs = GridSearchCV(Ridge(), {'alpha': alphas}, cv=kf5,
                         scoring='neg_root_mean_squared_error')
ridge_gs.fit(X_train_sc, y_train)
best_ridge_alpha = ridge_gs.best_params_['alpha']
print(f'Best Ridge alpha: {best_ridge_alpha}')

evaluate_model('Ridge', Ridge(alpha=best_ridge_alpha),
               X_train_sc, y_train, X_test_sc, y_test, yr_test)

# ── Lasso ──────────────────────────────────────────────────────────────────────
lasso_gs = GridSearchCV(Lasso(max_iter=5000), {'alpha': alphas}, cv=kf5,
                         scoring='neg_root_mean_squared_error')
lasso_gs.fit(X_train_sc, y_train)
best_lasso_alpha = lasso_gs.best_params_['alpha']
print(f'Best Lasso alpha: {best_lasso_alpha}')

lasso_final = evaluate_model('Lasso', Lasso(alpha=best_lasso_alpha, max_iter=5000),
                              X_train_sc, y_train, X_test_sc, y_test, yr_test)

In [ ]:
# Lasso feature selection: which weights did it set to exactly zero?
lasso_model = model_results['Lasso']['_model']
coef_df = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Coefficient': lasso_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

n_zero    = (coef_df['Coefficient'] == 0).sum()
n_nonzero = (coef_df['Coefficient'] != 0).sum()
print(f'Lasso zeroed out {n_zero} features and kept {n_nonzero}.')
print('Top 10 features kept by Lasso:')
print(coef_df[coef_df['Coefficient'] != 0].head(10).to_string(index=False))
print()
print('→ Positive coefficient: higher value → larger predicted fire')
print('→ Negative coefficient: higher value → smaller predicted fire')

### 6.4 — Support Vector Regressor (SVR)

SVR tries to fit the data within a **tolerance band** (like a highway). Points inside the band are considered "good enough" and don't affect the model — only points outside the band matter. This makes SVR inherently robust to small errors.

Key parameters:
- **C**: how much to penalise points that fall outside the band (higher C = stricter fit)
- **epsilon**: the width of the tolerance band (higher = more tolerance)

> **Historical note**: The original research paper on this exact dataset found that an SVR model with just 4 weather features (temp, RH, wind, rain) achieved the best Mean Absolute Deviation. We will see how it compares here.

In [ ]:
param_grid_svr = {
    'C':       [0.1, 1.0, 5.0, 10.0, 50.0],
    'epsilon': [0.1, 0.2, 0.5, 1.0],
}

svr_gs = RandomizedSearchCV(
    SVR(kernel='rbf'), param_grid_svr, n_iter=12,
    cv=kf5, scoring='neg_root_mean_squared_error',
    random_state=RANDOM_STATE
)
svr_gs.fit(X_train_sc, y_train)
best_svr = svr_gs.best_params_
print(f'Best SVR params: C={best_svr["C"]}, epsilon={best_svr["epsilon"]}')

evaluate_model('SVR', SVR(kernel='rbf', **best_svr),
               X_train_sc, y_train, X_test_sc, y_test, yr_test)

### 6.5 — Random Forest Regressor

A Random Forest builds **hundreds of decision trees in parallel**, each trained on a random sample of the training data and a random subset of features. The final prediction is the **average** across all trees.

Why is this better than a single tree? Each tree sees slightly different data, so it learns slightly different patterns. Averaging many different perspectives cancels out individual errors — a concept called **bagging** (bootstrap aggregating).

In [ ]:
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth':    [4, 6, 8, None],
    'max_features': ['sqrt', 0.5, 0.7],
}

rf_gs = RandomizedSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE),
    rf_params, n_iter=15, cv=kf5,
    scoring='neg_root_mean_squared_error',
    random_state=RANDOM_STATE
)
rf_gs.fit(X_train, y_train)
best_rf_params = rf_gs.best_params_
print(f'Best Random Forest params: {best_rf_params}')

rf_final = evaluate_model(
    'Random Forest',
    RandomForestRegressor(**best_rf_params, random_state=RANDOM_STATE),
    X_train, y_train, X_test, y_test, yr_test
)

In [ ]:
# Random Forest feature importances (mean decrease in impurity across all trees)
rf_model = model_results['Random Forest']['_model']
fi_df = pd.DataFrame({
    'Feature':    FEATURE_COLS,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(fi_df['Feature'][::-1], fi_df['Importance'][::-1],
        color='steelblue', edgecolor='white')
ax.set_title('Random Forest — Top 15 Feature Importances\n'
             '(Higher = more influential in the trees\' splits)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Importance (mean decrease in impurity)')
plt.tight_layout()
plt.show()

print('→ These importances reflect how often a feature is used for splitting and how much')
print('  it reduces prediction error when it is used.')
print('  Note: they do NOT tell us the direction of the effect (SHAP will do that in Section 7).')

### 6.6 — Gradient Boosted Trees (GBT)

Like Random Forest, GBT builds many decision trees. The key difference is **how** they are built:

- **Random Forest**: builds all trees **in parallel**, independently, then averages
- **GBT**: builds trees **one after another** — each new tree focuses specifically on the *errors* of the previous trees

This sequential error-correction is called **boosting**. It can lead to very accurate models, but is also more prone to overfitting than Random Forest if not tuned carefully.

In [ ]:
gbt_params = {
    'n_estimators':  [100, 200, 300],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth':     [3, 4, 5],
    'subsample':     [0.7, 0.8, 1.0],
}

gbt_gs = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=RANDOM_STATE),
    gbt_params, n_iter=15, cv=kf5,
    scoring='neg_root_mean_squared_error',
    random_state=RANDOM_STATE
)
gbt_gs.fit(X_train, y_train)
print(f'Best GBT params: {gbt_gs.best_params_}')

evaluate_model(
    'GBT',
    GradientBoostingRegressor(**gbt_gs.best_params_, random_state=RANDOM_STATE),
    X_train, y_train, X_test, y_test, yr_test
)

### 6.7 — XGBoost

**XGBoost** (*eXtreme Gradient Boosting*) is a highly optimised, faster version of Gradient Boosted Trees. It adds:
- Built-in **regularisation** (`reg_alpha`, `reg_lambda`) to prevent overfitting
- **Subsampling** of rows and columns per tree, adding diversity
- **Early stopping**: stop adding trees when the dev-set error stops improving

XGBoost has won hundreds of machine learning competitions and is one of the most widely used models in practice.

In [ ]:
# Train XGBoost with early stopping monitored on the dev set
xgb_model = xgb.XGBRegressor(
    n_estimators       = 1000,    # maximum trees; early stopping will cut this down
    learning_rate      = 0.05,    # smaller = more cautious, needs more trees
    max_depth          = 4,       # depth of each tree
    subsample          = 0.8,     # fraction of rows sampled per tree
    colsample_bytree   = 0.8,     # fraction of columns sampled per tree
    reg_alpha          = 0.1,     # L1 regularisation (like Lasso)
    reg_lambda         = 1.0,     # L2 regularisation (like Ridge)
    random_state       = RANDOM_STATE,
    early_stopping_rounds = 50,   # stop if dev RMSE hasn't improved for 50 rounds
    eval_metric        = 'rmse',
    verbosity          = 0
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_dev, y_dev)],
    verbose=False
)

print(f'Early stopping: best iteration was tree #{xgb_model.best_iteration}')
print(f'(training stopped before 1000 trees — saved time and prevented overfitting)')

In [ ]:
# Learning curves: how does dev RMSE improve as more trees are added?
evals_result = xgb_model.evals_result()
train_rmse_curve = evals_result['validation_0']['rmse']

# Re-train with eval on both train and dev to get both curves
xgb_curve = xgb.XGBRegressor(
    n_estimators=xgb_model.best_iteration + 1,
    learning_rate=0.05, max_depth=4, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=RANDOM_STATE, eval_metric='rmse', verbosity=0
)
xgb_curve.fit(X_train, y_train,
              eval_set=[(X_train, y_train), (X_dev, y_dev)],
              verbose=False)
res = xgb_curve.evals_result()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(res['validation_0']['rmse'], label='Train RMSE', color='steelblue', linewidth=1.5)
ax.plot(res['validation_1']['rmse'], label='Dev RMSE',   color='coral',     linewidth=1.5)
ax.axvline(xgb_model.best_iteration, color='green', linestyle='--', alpha=0.7,
           label=f'Best iteration ({xgb_model.best_iteration})')
ax.set_xlabel('Number of Trees (Boosting Rounds)')
ax.set_ylabel('RMSE (log scale)')
ax.set_title('XGBoost Learning Curves\n'
             'Early stopping prevents overfitting when dev RMSE stops improving',
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate the final XGBoost on the test set
xgb_test_pred_log = xgb_model.predict(X_test)
xgb_test_pred_ha  = np.expm1(xgb_test_pred_log).clip(min=0)

xgb_rmse = np.sqrt(mean_squared_error(yr_test, xgb_test_pred_ha))
xgb_mae  = mean_absolute_error(yr_test, xgb_test_pred_ha)
xgb_r2   = r2_score(yr_test, xgb_test_pred_ha)

# 5-fold CV score for consistency with other models
xgb_for_cv = xgb.XGBRegressor(
    n_estimators=xgb_model.best_iteration + 1,
    learning_rate=0.05, max_depth=4, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=RANDOM_STATE, verbosity=0
)
cv_xgb = -cross_val_score(xgb_for_cv, X_train, y_train, cv=kf5,
                            scoring='neg_root_mean_squared_error')

model_results['XGBoost'] = {
    'CV RMSE (log)':  f'{cv_xgb.mean():.3f} ± {cv_xgb.std():.3f}',
    'Test RMSE (ha)': round(xgb_rmse, 2),
    'Test MAE (ha)':  round(xgb_mae,  2),
    'R2':             round(xgb_r2,   3),
    '_model':         xgb_model,
    '_preds':         xgb_test_pred_ha,
}

print('──────────────────────────────────────────────────')
print('XGBoost')
print(f'  5-Fold CV RMSE (log) : {cv_xgb.mean():.4f} ± {cv_xgb.std():.4f}')
print(f'  Test RMSE (ha)       : {xgb_rmse:.2f}')
print(f'  Test MAE  (ha)       : {xgb_mae:.2f}')
print(f'  R²                   : {xgb_r2:.3f}')

### 6.8 — Model Comparison

Time to see all models side by side. Remember: **MAE is the most meaningful metric** here because RMSE is dominated by the single 1090-ha outlier.

In [ ]:
# Build the comparison table
display_keys = ['CV RMSE (log)', 'Test RMSE (ha)', 'Test MAE (ha)', 'R2']
results_table = pd.DataFrame({
    name: {k: v for k, v in metrics.items() if not k.startswith('_')}
    for name, metrics in model_results.items()
}).T
results_table.index.name = 'Model'

print('=== MODEL COMPARISON TABLE ===')
print(results_table.to_string())

In [ ]:
# Bar chart: Test MAE by model
mae_values = {name: m['Test MAE (ha)'] for name, m in model_results.items()}
mae_series = pd.Series(mae_values).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# MAE
colors = ['steelblue' if name != 'Naive Mean' else 'lightgray'
          for name in mae_series.index]
bars = axes[0].barh(mae_series.index, mae_series.values,
                    color=colors, edgecolor='white')
axes[0].set_title('Test MAE (hectares) — Lower is Better',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Mean Absolute Error (ha)')
for bar, val in zip(axes[0].patches, mae_series.values):
    axes[0].text(val + 0.2, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', fontsize=9)

# Predicted vs actual for best model (lowest MAE)
best_name = mae_series.index[0]
best_preds = model_results[best_name]['_preds']
axes[1].scatter(yr_test, best_preds, alpha=0.5, color='steelblue', edgecolors='none', s=30)
max_val = max(yr_test.max(), best_preds.max())
axes[1].plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Perfect prediction')
axes[1].set_xlabel('Actual area (ha)')
axes[1].set_ylabel('Predicted area (ha)')
axes[1].set_title(f'Best Model ({best_name})\nPredicted vs Actual',
                  fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Best model by MAE: {best_name} ({mae_series.iloc[0]:.2f} ha average error)')
print()
print('Key takeaway: all models struggle with the large fires.')
print('The predicted-vs-actual plot shows points cluster near 0 (most fires are small)')
print('but the few large fires are systematically under-predicted — this is the hardest part.')

---
## Section 7 — Model Explainability with SHAP

Knowing that XGBoost predicts "3.5 ha" is useful. Knowing **why** it predicts that is far more useful — especially for fire departments making operational decisions.

**SHAP** (*SHapley Additive exPlanations*) is the gold standard for explaining machine learning predictions. The core idea comes from game theory: it assigns each feature a "credit score" for the final prediction, fairly distributing the responsibility.

- A **positive SHAP value** for a feature means it pushed the prediction **higher** (more hectares)
- A **negative SHAP value** means it pushed the prediction **lower** (fewer hectares)
- SHAP values **sum to the difference** between the prediction and the model's average prediction

In [ ]:
# Compute SHAP values for the XGBoost model on the test set
# TreeExplainer is optimised for tree-based models — much faster than the general explainer

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

print(f'SHAP values computed for {len(X_test)} test samples x {shap_values.shape[1]} features')
print(f'Expected value (model baseline): {explainer.expected_value:.4f} log units')
print(f'  = {np.expm1(explainer.expected_value):.2f} ha (average prediction before any features)')

In [ ]:
# ── Global Beeswarm Plot ───────────────────────────────────────────────────────
# Each dot = one test sample
# X-axis = SHAP value (impact on prediction)
# Colour = the actual feature value (red=high, blue=low)
# Features are sorted by mean absolute SHAP value (most important at top)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test,
                  feature_names=FEATURE_COLS,
                  plot_type='dot',
                  max_display=15,
                  show=False)
plt.title('SHAP Summary — Global Feature Impact on XGBoost Predictions\n'
          'Red = high feature value, Blue = low feature value',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('How to read this plot:')
print('  • Each row is one feature; most important features appear at the top.')
print('  • Each dot is one fire in the test set.')
print('  • Dots to the RIGHT mean that feature increased the predicted area.')
print('  • Dots to the LEFT mean that feature decreased the predicted area.')
print('  • Red dots = high feature value; Blue dots = low feature value.')
print('  Example: if red (high temp) dots are to the right, it means higher temperature')
print('  pushes predictions toward larger fires — physically intuitive!')

In [ ]:
# ── Mean Absolute SHAP — overall feature ranking ──────────────────────────────
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.DataFrame({
    'Feature':    FEATURE_COLS,
    'Mean |SHAP|': mean_abs_shap
}).sort_values('Mean |SHAP|', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(shap_importance['Feature'][::-1],
        shap_importance['Mean |SHAP|'][::-1],
        color='steelblue', edgecolor='white')
ax.set_title('Mean Absolute SHAP Value — Feature Importance Ranking\n'
             '(Higher = feature has bigger average impact on predictions)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Mean |SHAP value| (in log-area units)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Dependence Plots — how does one feature affect predictions? ────────────────
# The colour of each dot shows the value of a second interacting feature
top3_features = shap_importance['Feature'].head(3).tolist()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, feat in zip(axes, top3_features):
    feat_idx = FEATURE_COLS.index(feat)
    shap.dependence_plot(
        feat_idx, shap_values, X_test,
        feature_names=FEATURE_COLS,
        ax=ax, show=False
    )
    ax.set_title(f'SHAP Dependence: {feat}', fontweight='bold')

plt.suptitle('How Do the Top Features Affect Predictions?\n'
             'Upward trend = higher value → larger predicted fire',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('How to read these plots:')
print('  • X-axis: actual value of the feature in the test set')
print('  • Y-axis: SHAP value for that feature (its contribution to the prediction)')
print('  • Colour: value of a second feature that interacts with this one')
print('  • An upward trend means: higher feature value → larger predicted burned area')

In [ ]:
# ── Local Explanations — explaining individual predictions ─────────────────────
# We pick the largest fire in the test set and a zero-fire record

largest_idx = yr_test.values.argmax()
zero_idx    = np.where(yr_test.values == 0)[0][0]

print('=== Explaining the Largest Fire in the Test Set ===')
print(f'Actual area:    {yr_test.iloc[largest_idx]:.2f} ha')
print(f'Predicted area: {np.expm1(xgb_model.predict(X_test.iloc[[largest_idx]])[0]):.2f} ha')
print(f'Model baseline: {np.expm1(explainer.expected_value):.2f} ha')
print()

# Show top contributing features for this specific prediction
sample_shap = shap_values[largest_idx]
sample_feats = pd.DataFrame({
    'Feature':     FEATURE_COLS,
    'Value':       X_test.iloc[largest_idx].values,
    'SHAP Impact': sample_shap
}).sort_values('SHAP Impact', key=abs, ascending=False).head(10)

print('Top 10 features driving this prediction (sorted by impact):')
print(sample_feats.to_string(index=False))
print()
print('Positive SHAP → pushed prediction UP (toward more hectares)')
print('Negative SHAP → pushed prediction DOWN (toward fewer hectares)')

In [ ]:
# Waterfall plot: visualise how features add up to the final prediction
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, idx, title in [
    (axes[0], largest_idx, f'Largest Fire ({yr_test.iloc[largest_idx]:.0f} ha actual)'),
    (axes[1], zero_idx,    f'Zero-Area Fire ({yr_test.iloc[zero_idx]:.0f} ha actual)'),
]:
    sample_sv = shap_values[idx]
    top_k     = 8
    top_idx   = np.argsort(np.abs(sample_sv))[::-1][:top_k]
    other_sum = sample_sv.sum() - sample_sv[top_idx].sum()

    feat_names  = [FEATURE_COLS[i] for i in top_idx] + ['All others']
    shap_vals   = list(sample_sv[top_idx]) + [other_sum]
    colors      = ['coral' if v > 0 else 'steelblue' for v in shap_vals]

    ax.barh(feat_names[::-1], shap_vals[::-1], color=colors[::-1], edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('SHAP value (log-area contribution)')
    ax.set_title(title, fontweight='bold', fontsize=11)
    pred_ha = np.expm1(xgb_model.predict(X_test.iloc[[idx]])[0])
    ax.text(0.02, 0.02, f'Predicted: {pred_ha:.1f} ha',
            transform=ax.transAxes, fontsize=9, color='navy')

plt.suptitle('Local SHAP Explanations — What drove each individual prediction?\n'
             'Red = pushed prediction UP | Blue = pushed DOWN',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('These two charts give us the model\'s "reasoning" for two contrasting cases.')
print('A fire department could use these to understand: for this specific fire,')
print('what conditions made the model predict a large/small spread?')

---
## Section 8 — Bonus: The Two-Stage Model

### Why a single regression model struggles here

About **48% of fires burn exactly 0 hectares** (the fire started but did not spread). When we ask a single regression model to predict area, it must simultaneously answer two very different questions:
1. *Will this fire spread at all?* (a yes/no question)
2. *If yes, how many hectares?* (a numeric question)

A **two-stage model** separates these concerns:

```
Stage 1: Classifier  → "Did the fire spread?" (0 vs >0)
                              │
               ┌─────────────┴──────────────┐
              No (predict 0)           Yes (go to Stage 2)
                                             │
                                   Stage 2: Regressor
                                   → "How many hectares?"
```

This does not always beat a single model in practice (shown below), but it is conceptually the correct approach for **zero-inflated** targets.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Create binary labels: 0 = no spread, 1 = spread
y_train_clf = (yr_train > 0).astype(int)
y_test_clf  = (yr_test  > 0).astype(int)

# Stage 1: Classifier — does the fire spread?
clf = RandomForestClassifier(n_estimators=200, max_depth=6,
                              class_weight='balanced',  # handles the class imbalance
                              random_state=RANDOM_STATE)
clf.fit(X_train, y_train_clf)
clf_preds = clf.predict(X_test)

print('=== Stage 1: Classifier — Fire vs No Fire ===')
print(classification_report(y_test_clf, clf_preds,
                             target_names=['No spread (0 ha)', 'Spread (>0 ha)']))
print('class_weight=balanced helps because no-spread fires dominate the dataset.')
print('Without it, the classifier would just predict "no spread" for everything.')

In [ ]:
# Stage 2: Regressor — trained only on fires that actually spread
spread_mask_train = yr_train > 0

reg_stage2 = xgb.XGBRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=4,
    random_state=RANDOM_STATE, verbosity=0
)
reg_stage2.fit(X_train[spread_mask_train], y_train[spread_mask_train])

# ── Combine Stage 1 + Stage 2 predictions ─────────────────────────────────────
# For fires predicted to not spread → predict 0 ha
# For fires predicted to spread     → use the Stage 2 regressor

two_stage_preds = np.zeros(len(X_test))
spread_predicted = clf_preds == 1

if spread_predicted.sum() > 0:
    reg_preds = np.expm1(reg_stage2.predict(X_test[spread_predicted])).clip(min=0)
    two_stage_preds[spread_predicted] = reg_preds

ts_rmse = np.sqrt(mean_squared_error(yr_test, two_stage_preds))
ts_mae  = mean_absolute_error(yr_test, two_stage_preds)
ts_r2   = r2_score(yr_test, two_stage_preds)

print('=== Two-Stage Model Results ===')
print(f'  Test RMSE (ha) : {ts_rmse:.2f}')
print(f'  Test MAE  (ha) : {ts_mae:.2f}')
print(f'  R²             : {ts_r2:.3f}')
print()

# Compare to single XGBoost
single_mae = model_results['XGBoost']['Test MAE (ha)']
improvement = single_mae - ts_mae
print(f'Single XGBoost MAE: {single_mae:.2f} ha')
print(f'Two-Stage MAE:      {ts_mae:.2f} ha')
print(f'Difference:         {improvement:+.2f} ha ("+" = two-stage is better)')
print()
print('Conclusion:')
print('The two-stage approach is conceptually sound for zero-inflated targets.')
print('Whether it outperforms a single model depends on how well Stage 1 classifies fires.')
print('With this small dataset (517 rows), the classifier has limited data to learn from.')
print('With larger fire databases, the two-stage approach typically wins.')

---
## Conclusion

### What We Built

This notebook walked through a **complete, professional machine learning pipeline** from raw (deliberately corrupted) data to an explained XGBoost model:

| Step | What We Did |
|------|-------------|
| **Data Cleaning** | Removed duplicates, fixed typos, corrected types, capped outliers, imputed missing values with MICE |
| **EDA** | Histograms, correlation heatmaps, spatial maps, seasonal patterns |
| **Feature Engineering** | log(area+1) transform, one-hot + cyclic encoding, StandardScaler |
| **Splitting** | 70/10/20 train/dev/test split with stratification, 5-fold CV |
| **Modelling** | Naive baseline → Decision Tree → Ridge → Lasso → SVR → Random Forest → GBT → XGBoost |
| **Explainability** | SHAP global importance, dependence plots, local waterfall explanations |
| **Bonus** | Two-stage classifier + regressor for zero-inflated targets |

### Key Lessons from This Dataset

1. **Fire area prediction is genuinely hard.** Even the best models have weak correlations with the target. This reflects real physical uncertainty — fire spread depends on tiny local factors (topography, fuel gaps, exact wind gusts) not captured in our 13 features.

2. **The extreme outlier drives RMSE for all models.** Always report MAE alongside RMSE for skewed targets.

3. **Log-transforming the target is essential.** Without it, models would focus almost entirely on the handful of large fires.

4. **SHAP is your bridge between the model and the field team.** Instead of a black-box number, SHAP gives you: *"This fire was predicted to spread 12 ha mainly because DC was very high (long drought) and temperature exceeded 28°C."*

### Potential Next Steps

- Collect more data — 517 records is small for machine learning
- Add features: elevation, vegetation type, proximity to roads
- Try a neural network or a more sophisticated two-stage model
- Formulate as a classification task: *"Will this be a large fire (>10 ha)?"* rather than exact area prediction